## 2 Quadratic Program (QP)

Consider the convex quadratic program (QP) in the form

$$
\min_{x} \quad \phi = \frac{1}{2} x^\top H x + g^\top x \tag{2a}
$$

$$
\text{s.t.} \quad b_l \leq A^\top x \leq b_u, \tag{2b}
$$

$$
x_l \leq x \leq x_u. \tag{2c}
$$

### (a)
 What is the Lagrangian function of this problem (2)?

---

STEP 1: Firstly, we rewrite all the constraints in the standard inequality form:

$$
C^T x \ge d
$$

where

$$
C =
\begin{bmatrix}
A & -A & I & -I
\end{bmatrix},
\quad
d =
\begin{bmatrix}
b_l \\
- b_u \\
x_l \\
- x_u
\end{bmatrix}
$$

so that

$$
C^T x =
\begin{bmatrix}
A^T x \\
- A^T x \\
x \\
- x
\end{bmatrix}
$$

---

STEP 2: Secondly, we define the Lagrangian as 

$$
L(x,z) = \frac{1}{2} x^T H x + g^T x - z^T (C^T x - d)
$$

where

$$
z \ge 0
$$

are the Lagrange multipliers associated with the inequality constraints.

---


### (b) 
Write the necessary and sufficient optimality conditions for (2).

---

Since $H \succ 0$, the problem is strictly convex, and therefore the KKT conditions are both necessary and sufficient for optimality.



1. Stationarity

The Lagrangian is given by

$$
L(x,z) = \frac{1}{2} x^T H x + g^T x - z^T (C^T x - d)
$$

We compute the gradient with respect to \(x\):

$$
\nabla_x L(x,z) = \nabla_x \left( \frac{1}{2} x^T H x \right)
+ \nabla_x (g^T x)
- \nabla_x \left( z^T C^T x \right)
$$

Using standard results:

- Since \(H\) is symmetric,

$$
\nabla_x \left( \frac{1}{2} x^T H x \right) = Hx
$$

- For the linear term,

$$
\nabla_x (g^T x) = g
$$

- For the constraint term,

$$
\nabla_x \left( z^T C^T x \right) = Cz
$$

Thus, the stationarity condition becomes

$$
\nabla_x L(x,z) = Hx + g - Cz = 0
$$



2. Primal Feasibility

$$
C^T x - d \ge 0
$$



3. Dual Feasibility

$$
z \ge 0
$$



4. Complementarity

$$
z_i (C^T x - d)_i = 0, \quad \forall i
$$



Introducing slack variables \(s = C^T x - d\), the KKT conditions can be written as

$$
\begin{aligned}
r_L &= Hx + g - Cz = 0 \\
r_C &= -C^T x + s + d = 0 \\
(z,s) &\ge 0 \\
SZe &= 0
\end{aligned}
$$

---

### (c)
 Generate code for creating test problems in the form (2). Describe the code.

---


We generate random convex quadratic programming (QP) problems of the form

$$
\min_x \ \frac{1}{2} x^T H x + g^T x
$$

s.t.

$$
b_l \le A^T x \le b_u, \quad l \le x \le u
$$
 
We mostly follow the implementation in course materials 6.1. (slide 18 & 19) but adjust it to python.
Test problems are generated by constructing the matrices $ H $, $ g $, $ A $ and the bounds $ b_l $, $ b_u $, $ x_l $, $ x_u $.
The Hessian $ H $ must be symmetric positive definite. This can be ensured by defining

$
H = R^T R + \alpha I
$

where \(R\) is a randomly generated matrix.

The remaining matrices and vectors are generated randomly.

The number of variables is \(x \in \mathbb{R}^n\). The number of constraints is chosen as

$$
m = \text{round}(\beta n), \quad 0 < \beta < 1
$$

so that the size of the problem depends on $n$.


The constraint matrix $ A \in \mathbb{R}^{n \times m}$ is generated as a sparse matrix with a specified density. The nonzero elements are drawn from a normal distribution

$$
A_{ij} \sim \mathcal{N}(0,1)
$$

Similarly, a sparse matrix $ M \in \mathbb{R}^{n \times n} $ is generated with elements

$$
M_{ij} \sim \mathcal{N}(0,1)
$$

The Hessian is constructed as

$$
H = M M^T + \alpha I
$$

which ensures that $ H $ is symmetric positive definite and the quadratic program is strictly convex.


The linear term is generated as

$$
g_i \sim \mathcal{U}([-1,1])
$$

The bounds are generated independently as

$$
b_l, \; b_u \sim \mathcal{U}([-1,1])
$$

The variable bounds are chosen as

$$
l = -\mathbf{1}, \quad u = \mathbf{1}
$$


For use in interior-point algorithms, the constraints are rewritten in the standard form

$$
C^T x \ge d
$$

where

$$
C =
\begin{bmatrix}
A & -A & I & -I
\end{bmatrix},
\quad
d =
\begin{bmatrix}
b_l \\
- b_u \\
l \\
- u
\end{bmatrix}
$$

The implementation allows both sparse and dense representations depending on a flag, similar to the interface used in Task 1.

---

We define all imports for the implementations.

In [89]:
import cvxpy as cp
import numpy as np
import time
from utils import SolutionStats

import sys
import os

sys.path.append(os.path.abspath(".."))

In [90]:
import scipy.sparse as sp

class RandomQPGenerator:
    """
    Generates data for a random convex QP

    min (1/2)x^T H x + g^T x
    s.t. bl <= A^T x <= bu
        l <= x <= u

    Parameters:
        n       : number of variables
        alpha   : regularization factor (alpha > 0)
        density : density of sparse matrices (0 < density < 1)

    Returns:
        H, g, bl, A, bu, l, u
        H and A are sparse matrices
    """
    def __init__(self, n, alpha, density, beta, flag="sparse"):
        self.n = n
        self.alpha = alpha
        self.density = density
        self.beta = beta
        self.flag = flag

        self.H = None
        self.g = None
        self.bl = None
        self.A = None
        self.bu = None
        self.l = None
        self.u = None

        self.C = None
        self.d = None
        self.A_eq = None
        self.b_eq = None

    def generate_interior_point_form(self, H, g, bl, A, bu, l, u):
        n, m = A.shape

        # Build C — keep sparse or dense to match A
        if sp.issparse(A):
            I = sp.eye(n, format='csr')
            self.C = sp.hstack([A, -A, I, -I], format='csr')
        else:
            self.C = np.hstack([A, -A, np.eye(n), -np.eye(n)])

        # Build d
        self.d = np.vstack([
            bl,
            -bu,
            l,
            -u
        ]).flatten()

        # No equality constraints
        self.A_eq = np.zeros((n, 0))
        self.b_eq = np.zeros(0)


    def generate(self):
        """
        Returns:
            H, g, bl, A, bu, l, u - we call this the "general" form
            H, g, A_eq, b_eq, C, d - we call this the "interior-point" form
            H and A are sparse matrices
        """

        m = round(self.beta * self.n)

        # Sparse A ~ N(0,1)
        self.A = sp.random(self.n, m, density=self.density, format='csr', data_rvs=np.random.randn)

        # Sparse M
        M = sp.random(self.n, self.n, density=self.density, format='csr', data_rvs=np.random.randn)

        # H = M M^T + alpha I (positive definite)
        self.H = M @ M.T + self.alpha * sp.eye(self.n, format='csr')

        # Convert to dense immediately if requested
        if self.flag == 'dense':
            self.A = self.A.toarray()
            self.H = self.H.toarray()

        # Bounds
        self.bl = -np.random.rand(m, 1)     # U([-1,0])
        self.bu = np.random.rand(m, 1)      # U([0,1])

        # Linear term g ~ U([-1,1])
        self.g = np.random.uniform(-1, 1, (self.n, 1))

        # Variable bounds
        self.l = -np.ones((self.n, 1))
        self.u = np.ones((self.n, 1))

        self.generate_interior_point_form(self.H, self.g, self.bl, self.A, self.bu, self.l, self.u)
        

    def get_general_problem(self):
        return self.H, self.g, self.bl, self.A, self.bu, self.l, self.u
    
    def get_interior_point_problem(self):
        return self.H, self.g, self.A_eq, self.b_eq, self.C, self.d

### (d)
Solve the test problems using Matlab’s `quadprog` or/and other library QP solvers.  
Read `quadprog`’s documentation to make sure you pass the input correctly.  

You are welcome to (also) solve the problem with other optimization software, e.g. IPOPT with Casadi, JuMP with Gurobi in Julia, etc.  

Document this by providing code in the report. Report solution statistics (number of iterations, CPU time, etc).


---

First, we define a SoulutionStats class for keeping track of the performance metrics.

In [91]:
class QPsolver:

    def __init__(self, H, g, bl, A, bu, l, u):
        self.H = H
        self.g = g.flatten()
        self.bl = bl.flatten()
        self.A = A
        self.bu = bu.flatten()
        self.l = l.flatten()
        self.u = u.flatten()

    def __name__(self):
        return "CVXPY QP Solver"
    
    def solve(self):
        n = self.H.shape[0]

        # Define variable
        x = cp.Variable(n)

        # Objective 
        H_psd = cp.psd_wrap(self.H) #— psd_wrap skips the ARPACK PSD check for large sparse H 
                                    #since I decided to reuse the presentation code, which generates sparse H 
        objective = 0.5 * cp.quad_form(x, H_psd) + self.g @ x

        # Constraints
        constraints = [
            self.A.T @ x >= self.bl,
            self.A.T @ x <= self.bu,
            x >= self.l,
            x <= self.u
        ]

        # Problem
        prob = cp.Problem(cp.Minimize(objective), constraints)

        start = time.time()
        # solve problem
        prob.solve()
        end = time.time()

        x = x.value
        obj = 0.5 * x @ (self.H @ x) + self.g @ x

        return SolutionStats(
            x=x,
            iterations=prob.solver_stats.num_iters,
            time=end - start,
            obj=obj,
            feasibility=None 
        )

### (e)
Write pseudo-code for a primal active-set algorithm for solution of the problem (2).  

Explain the major steps in your algorithm. Explain how you find a feasible initial point.  

You can either convert the problem to a standard form (less efficient), or use the structure directly (more efficient).  

Discuss the pros and cons of the method you choose. You can also implement several versions and compare them by numerical experiments.


---

### (f)
Implement the primal active-set algorithm for (2) and test it.  

You must provide: commented code, driver files to test your code documentation that it works, performance statistics  

Test your software on your test problems.

---

In [92]:
from LP_solvers.LPsolver import LPsolver
from EqualityConstrainQP_solvers.KKTSolver_temporary import KKTSolver
import numpy as np
import time

class PrimalActiveSetSolver:

    def __init__(self, H, g, bl, A, bu, l, u):
        self.H = H
        self.g = g.flatten()
        self.bl = bl.flatten()
        self.A = A
        self.bu = bu.flatten()
        self.l = l.flatten()
        self.u = u.flatten()

        self.k = 0
        self.max_iterations = 1000

    def __name__(self):
        return "Primal Active Set Solver"
    
    def build_working_set(self, x, tol=1e-6):
        W = []

        Ax = self.A.T @ x

        # Linear constraints
        for i in range(len(Ax)):
            if abs(Ax[i] - self.bl[i]) <= tol:
                W.append(("bl", i))
            elif abs(Ax[i] - self.bu[i]) <= tol:
                W.append(("bu", i))

        # Box constraints
        for i in range(len(x)):
            if abs(x[i] - self.l[i]) <= tol:
                W.append(("xl", i))
            elif abs(x[i] - self.u[i]) <= tol:
                W.append(("xu", i))

        return W
    
    def build_A_W(self, W):
        rows = []

        for (ctype, i) in W:

            if ctype == "bl":
                # a_i^T x >= bl  →  row = +a_i
                row = self.A[:, i].toarray().flatten()
            elif ctype == "bu":
                # a_i^T x <= bu  →  rewrite as -a_i^T x >= -bu  →  row = -a_i
                row = -self.A[:, i].toarray().flatten()
            elif ctype == "xl":
                # x_i >= l_i  →  row = +e_i
                row = np.zeros(self.H.shape[0])
                row[i] = 1.0
            elif ctype == "xu":
                # x_i <= u_i  →  rewrite as -x_i >= -u_i  →  row = -e_i
                row = np.zeros(self.H.shape[0])
                row[i] = -1.0

            rows.append(row)

        if len(rows) == 0:
            return np.zeros((0, self.H.shape[0]))

        A_W = np.vstack(rows)

        return A_W

    def compute_alpha(self, x, p, W, tol=1e-6):
        alpha = 1.0
        hit_constraint = None

        n = len(x)
        m = self.A.shape[1]

        Ax = self.A.T @ x
        Ap = self.A.T @ p

        #Liniear constraints
        for i in range(m):

            #skip active constraints already in W
            if ("bl", i) in W or ("bu", i) in W:
                continue

            if Ap[i] > tol:
                alpha_i = (self.bu[i] - Ax[i]) / Ap[i]
                if alpha_i >= 0 and alpha_i < alpha:
                    alpha = alpha_i
                    hit_constraint = ("bu", i)

            elif Ap[i] < -tol:
                alpha_i = (self.bl[i] - Ax[i]) / Ap[i]
                if alpha_i >= 0 and alpha_i < alpha:
                    alpha = alpha_i
                    hit_constraint = ("bl", i)

        #Box constraints
        for i in range(n):

            #skip active constraints already in W
            if ("xl", i) in W or ("xu", i) in W:
                continue

            if p[i] > tol:
                alpha_i = (self.u[i] - x[i]) / p[i]
                if alpha_i >= 0 and alpha_i < alpha:
                    alpha = alpha_i
                    hit_constraint = ("xu", i)

            elif p[i] < -tol:
                alpha_i = (self.l[i] - x[i]) / p[i]
                if alpha_i >= 0 and alpha_i < alpha:
                    alpha = alpha_i
                    hit_constraint = ("xl", i)

        return alpha, hit_constraint
    
    def solve(self):
        start = time.time()
        # Init
        #find feasible starting point
        x = LPsolver(self.H, self.g, self.bl, self.A, self.bu, self.l, self.u).x

        #build working set W_0 - that includes all active constraints at the feasible starting point
        W = self.build_working_set(x)

        #Main loop
        while self.k < self.max_iterations:
            A_W = self.build_A_W(W)
            kkt_solver = KKTSolver()
            p, lambda_ = kkt_solver.solve(self.H, self.g, x, A_W)

            #Case 1: p = 0
            if np.linalg.norm(p) < 1e-6:
                #if all lambda >= 0 - all constaints in W are satisfied - we found a solution
                if np.all(lambda_ >= 0):
                    obj = 0.5 * x @ (self.H @ x) + self.g @ x
                    end = time.time()
                    return SolutionStats(
                        x=x,
                        iterations=self.k,
                        time=end - start,
                        obj=obj,
                        feasibility=None 
                    )
                else:
                    #remove the constraint with the  negative lambda from W and repeat (most penalized constraint)
                    neg_idx = np.where(lambda_ < -1e-8)[0]
                    j = neg_idx[0]
                    W.pop(j) # W_k+1 = W_k \ {penalized_constraint}
                             # x_k+1 = x_k
                    self.k += 1

            #Case 2: p != 0
            else:
                #we compute the max step we can take (alpha) following p direstion without violating a constraints inactive
                alpha, hit_constraint = self.compute_alpha(x, p, W) 

                #a constaint was hit
                if alpha < 1.0:
                    #we add the hit constraint to our active set and add it to W_k+1 and take the biggest step (alpha) we can 
                    if hit_constraint not in W:
                        W.append(hit_constraint) # W_k+1 = W_k + {hit_constraint}
                    x = x + alpha * p # x_k+1 = x_k + alpha * p

                #no constraint was hit - we can take the full step
                else:
                    x = x + p # x_k+1 = x_k + p 

                self.k += 1


        #no solution found within max iterations
        end = time.time()
        obj = 0.5 * x @ (self.H @ x) + self.g @ x
        return SolutionStats(
            x=None,
            iterations=self.k,
            time=end - start,
            obj=obj,
            feasibility=None 
        )

### (g)
Write pseudo-code for a primal-dual interior-point algorithm for solution of the problem (2).  

Explain the major steps in your algorithm. You should describe an algorithm tailored for the QP in the form (2).


---

### (h)
Implement the primal-dual interior-point algorithm for (2) and test it.  

You must provide: commented code, driver files to test your code documentation that it works, performance statistics 

Test your software on your test problems.

---

In [93]:
from scipy.linalg import solve
from scipy.sparse import issparse

class PrimalDualInteriorPointSolver:

    def __init__(self, H, g, A, b, C, d):
        self.H = H.toarray() if issparse(H) else np.asarray(H)
        self.g = g.flatten()
        self.A = A
        self.b = b.flatten()
        self.C = C
        self.d = d.flatten()

        self.k = 0
        self.max_iter = 1000
        self.tol = 1e-6
        self.ni = 0.995  # step size reduction factor to ensure we stay in the interior of the feasible region

    def __name__(self):
        return "Primal-Dual Interior Point Solver"        
    
    def initialize(self):
        #we follow the heuristic for an initial point from the lectures
        n = self.H.shape[0]
        m = self.A.shape[1]
        mc = self.C.shape[1]

        #same as in the primal dual interior point pdf
        x_bar = np.zeros(n)
        y_bar = np.zeros(m)
        z_bar = np.ones(mc)
        s_bar = np.ones(mc)

        #Compute residuals 
        rL, rA, rC, rSZ = self.compute_residuals(x_bar, y_bar, z_bar, s_bar)

        #compute affine direction using augmented system
        dx_aff, dy_aff, dz_aff, ds_aff = self.compute_newton_direction(
            rL, rA, rC, rSZ, x_bar, z_bar, s_bar
        )

        #enforce positivity
        z = np.maximum(1.0, np.abs(z_bar + dz_aff))
        s = np.maximum(1.0, np.abs(s_bar + ds_aff))
        x = x_bar + dx_aff
        y = y_bar + dy_aff

        return x, y, z, s
    
    def compute_residuals(self, x, y, z, s):
        #penalties for not satisfying the KKT conditions - we want to drive these to zero
        #they tell us how far we are form optimlatity
        rL = self.H @ x + self.g - self.A @ y - self.C @ z
        rA = self.b - self.A.T @ x
        rC = s + self.d - self.C.T @ x
        rSZ = s * z  #just rSZ[i] = s[i] * z[i] product of s and z - like we multiply the diagonal matrix of s with z or the diagonal matrix of z with s - it is the same

        return rL, rA, rC, rSZ
    
    def compute_mu(self, z, s):
        #measure distance to central path - we want to drive this to zero
        return (z @ s) / len(z)
    
    def check_convergence(self, rL, rA, rC, rSZ, mu):
        #check convergence conditions - if we are close enough to solution 
        if (np.linalg.norm(rL) < self.tol and 
            np.linalg.norm(rA) < self.tol and 
            np.linalg.norm(rC) < self.tol and 
            mu < self.tol):
            return True
        return False
    
    def compute_newton_direction(self, rL, rA, rC, rSZ, x, z, s, target=None):
        """Solve the augmented 2-block system to get Newton directions.

        Instead of the reduced normal equations (which require z/s ratios
        that blow up for near-active constraints), solve the larger but
        numerically stable augmented system:

            [H,     -C ] [dx]   =  [-rL                  ]
            [Z*C^T,  S ] [dz]      [Z*rC - rSZ + target  ]

        where Z = diag(z), S = diag(s).  No z/s ratio appears.
        dz is O(1) even when z/s → ∞.
        """
        if target is None:
            target = np.zeros_like(z)

        n  = len(x)
        mc = len(z)

        Z  = z          # diagonal entries of diag(z)
        S  = s          # diagonal entries of diag(s)

        # Build the (n + mc) x (n + mc) augmented system
        #   [H,      -C  ]
        #   [Z*C^T,   S  ]
        top_left  = self.H                     # n x n
        top_right = -self.C                    # n x mc
        bot_left  = (Z[:, None] * self.C.T)    # mc x n  (row-wise scale of C^T)
        bot_right = np.diag(S)                 # mc x mc

        A_aug = np.block([
            [top_left,  top_right],
            [bot_left,  bot_right]
        ])

        rhs_top = -rL
        rhs_bot = Z * rC - rSZ + target        # = z*rC - z*s + target
        rhs_aug = np.concatenate([rhs_top, rhs_bot])

        sol    = solve(A_aug, rhs_aug)
        dx     = sol[:n]
        dz     = sol[n:]

        # dy is zero (no equality constraints in this formulation; A is empty)
        dy = np.zeros(self.A.shape[1])

        # ds from primal feasibility: ds = C^T dx - rC
        ds = self.C.T @ dx - rC

        return dx, dy, dz, ds
    
    def compute_step_alpha(self, z, dz, s, ds):
        alpha_p = 1.0   # primal: governs s (and x, which is unconstrained but fine)
        alpha_d = 1.0   # dual:   governs z

        idx = ds < 0
        if np.any(idx):
            alpha_p = min(alpha_p, np.min(-s[idx] / ds[idx]))

        idx = dz < 0
        if np.any(idx):
            alpha_d = min(alpha_d, np.min(-z[idx] / dz[idx]))

        return alpha_p, alpha_d
    
    def solve(self):
        start = time.time()

        x, y, z, s = self.initialize()

        while self.k < self.max_iter :
            rL, rA, rC, rSZ = self.compute_residuals(x, y, z, s)
            mu = self.compute_mu(z, s)
            if self.check_convergence(rL, rA, rC, rSZ, mu):
                obj = 0.5 * x @ (self.H @ x) + self.g @ x
                end = time.time()
                return SolutionStats(
                    x=x,
                    iterations=self.k,
                    time=end - start,
                    obj=obj,
                    feasibility=None
                )

            #Predictor step — affine direction (no centering)
            dx_aff, dy_aff, dz_aff, ds_aff = self.compute_newton_direction(
                rL, rA, rC, rSZ, x, z, s, target=None
            )
            alpha_aff_p, alpha_aff_d = self.compute_step_alpha(z, dz_aff, s, ds_aff)

            #compute affine duality gap (split step)
            mu_aff = ((z + alpha_aff_d * dz_aff) @ (s + alpha_aff_p * ds_aff)) / len(z)
            mu_aff = max(mu_aff, 0.0)
            #compute centering parameter
            sigma = np.clip((mu_aff / mu)**3, 0.0, 1.0)

            #Corrector step — Mehrotra centering + second-order correction
            # dz_aff and ds_aff are O(1) (augmented system avoids z/s blowup)
            # so the cross product dz_aff*ds_aff is bounded and the correction is valid
            target_corr = sigma * mu - dz_aff * ds_aff

            dx, dy, dz, ds = self.compute_newton_direction(
                rL, rA, rC, rSZ, x, z, s, target=target_corr
            )
            alpha_p, alpha_d = self.compute_step_alpha(z, dz, s, ds)

            # we take a fraction of the step to ensure we stay in the interior of the feasible region
            alpha_p = self.ni * alpha_p
            alpha_d = self.ni * alpha_d

            #update iteration
            x = x + alpha_p * dx
            y = y + alpha_d * dy
            z = z + alpha_d * dz
            s = s + alpha_p * ds

            self.k += 1

        #no solution found within max iterations
        end = time.time()
        obj = 0.5 * x @ (self.H @ x) + self.g @ x
        return SolutionStats(
            x=x,
            iterations=self.k,
            time=end - start,
            obj=obj,
            feasibility=None 
        )

### (i)
Compare:
- library QP solvers  
- active-set QP solver(s)  
- primal-dual QP solver  

Discuss performance and provide tables and figures illustrating the results.

---

In [94]:
from random_qp import RandomQPGenerator
from QP_solvers.QPsolver import QPsolver
from QP_solvers.PrimalActiveSet import PrimalActiveSetSolver
from QP_solvers.PrimalDualInteriorPoint import PrimalDualInteriorPointSolver
from utils import SolutionStats
from visualiser import plot_runtime_vs_n, plot_iterations_vs_n

def run_experiment(n_values, density=0.1, alpha=1e-2, beta=0.5):
    results = {
        "cvxpy": [],
        "active_set": [],
        "interior_point": []
    }

    for n in n_values:
        print(f"\nRunning problem with n = {n}")

        generator = RandomQPGenerator(n, alpha, density, beta=0.3, flag="sparse")
        generator.generate()

        # General problem
        H, g, bl, A, bu, l, u = generator.get_general_problem()

        # Interior-point form
        H_ip, g_ip, A_eq, b_eq, C, d = generator.get_interior_point_problem()

        solvers = {
            "cvxpy": QPsolver(H, g, bl, A, bu, l, u),
            "active_set": PrimalActiveSetSolver(H, g, bl, A, bu, l, u),
            "interior_point": PrimalDualInteriorPointSolver(H_ip, g_ip, A_eq, b_eq, C, d)
        }

        for name, solver in solvers.items():
            stats = solver.solve()

            results[name].append(stats)

            print(f"{name}: obj={stats.obj:.4f}, time={stats.time:.4f}, iter={stats.iterations}")

    return results




n_values = [20, 50, 100, 200]

results = run_experiment(n_values)

plot_runtime_vs_n(results, n_values)
plot_iterations_vs_n(results, n_values)





Running problem with n = 20
cvxpy: obj=-5.1137, time=0.0220, iter=100
active_set: obj=-5.1137, time=0.0371, iter=16
interior_point: obj=-5.1137, time=0.0074, iter=6

Running problem with n = 50
cvxpy: obj=-6.5197, time=0.0148, iter=100
active_set: obj=-6.5197, time=0.0652, iter=33
interior_point: obj=-6.5197, time=0.0208, iter=6

Running problem with n = 100
cvxpy: obj=-6.7527, time=0.0224, iter=75


KeyboardInterrupt: 